### 工作進度  
* 【置頂】**筆記內容架構**與**量化技術分析系統**相關資訊請參閱[260315筆記.ipynb](https://github.com/yilintung/StockInvestmentNotebook/blob/main/260315%E7%AD%86%E8%A8%98.ipynb)之「工作進度」。
* 本日延續[260319筆記.ipynb](https://github.com/yilintung/StockInvestmentNotebook/blob/main/260319%E7%AD%86%E8%A8%98.ipynb)的「技術開發」，進行相關架構的調整與修改。詳細請參照ChatGPT對話[紀錄](https://chatgpt.com/share/69bffe28-8d98-800c-9860-9029155689a9)。  

In [1]:
import os
import pandas as pd
import numpy as np
import datetime
import sqlite3
import io
import base64
import matplotlib.pyplot as plt
import matplotlib
import mplfinance as mpf
import markdown

import json

from talib.abstract import *
from dotenv import load_dotenv, find_dotenv
from PIL import Image
from IPython.core.display import HTML

from typing import Any, Dict, List, Optional, Tuple
from openai import OpenAI

In [2]:
##### 【公用函式】 來源 ： https://www.cnblogs.com/Rosaany/p/15561918.html #####
def get_monday_to_sunday(today, weekly=0):
    last = weekly * 7
    today = datetime.datetime.strptime(str(today), "%Y-%m-%d")
    monday = datetime.datetime.strftime(today - datetime.timedelta(today.weekday() - last), "%Y-%m-%d")
    monday_ = datetime.datetime.strptime(monday, "%Y-%m-%d")
    sunday = datetime.datetime.strftime(monday_ + datetime.timedelta(monday_.weekday() + 6), "%Y-%m-%d")
    return monday, sunday

###### 【內部函式】 當日Ｋ價格資料為零時之修正函式 ######
def correcting_zero_price_issue( daily_price_df, debug = False) :
    # 開啟寫時複製(Copy-on-Write)
    copy_on_write = pd.options.mode.copy_on_write
    pd.options.mode.copy_on_write = True
    
    # 找出開盤價、最高價、收盤價與最低價皆為0的價格資料
    zero_prices_df =  daily_price_df[(daily_price_df['Open'] == 0.0) & (daily_price_df['High'] == 0.0) & (daily_price_df['Low'] == 0.0) & (daily_price_df['Close'] == 0.0)]
    if zero_prices_df.empty is False :
        zero_prices_idx = zero_prices_df.index
        df_first_idx    = daily_price_df.iloc[0].name
        for idx in zero_prices_idx :
            if (idx - 1) >= df_first_idx :
                # 當開盤價、最高價、收盤價與最低價皆為0時，會用前一個交易日的收盤價來做修正
                prev_close_price = daily_price_df.loc[idx-1]['Close']
                prev_stock_id    = daily_price_df.loc[idx-1]['StockID']
                curr_stock_id    = daily_price_df.loc[idx]['StockID']
                if debug is True :
                    print('ＤＥＢＵＧ ： 〈代碼：{}，日期：{}〉  修改前：開 ＝ {} 高 ＝ {} 低 ＝ {} 收 ＝ {}'.format(curr_stock_id,daily_price_df.loc[idx]['Date'],daily_price_df.loc[idx,'Open'],daily_price_df.loc[idx,'High'],daily_price_df.loc[idx,'Low'],daily_price_df.loc[idx,'Close']), end='')
                daily_price_df.loc[idx,'Open']  = prev_close_price
                daily_price_df.loc[idx,'High']  = prev_close_price
                daily_price_df.loc[idx,'Low']   = prev_close_price
                daily_price_df.loc[idx,'Close'] = prev_close_price
                if debug is True :
                    print(' ， 修改後：開 ＝ {} 高 ＝ {} 低 ＝ {} 收 ＝ {}'.format(daily_price_df.loc[idx,'Open'],daily_price_df.loc[idx,'High'],daily_price_df.loc[idx,'Low'],daily_price_df.loc[idx,'Close']))
            else :
                pass
    
    # 還原初始狀態
    pd.options.mode.copy_on_write = copy_on_write

###### 【公用函式】 來源 ： https://stackoverflow.com/questions/73604477/i-am-trying-to-crop-an-image-to-remove-extra-space-in-python #####
def get_first_last(mask, axis: int):
    """ Find the first and last index of non-zero values along an axis in `mask` """
    mask_axis = np.argmax(mask, axis=axis) > 0
    a = np.argmax(mask_axis)
    b = len(mask_axis) - np.argmax(mask_axis[::-1])
    return int(a), int(b)

def crop_borders(img, crop_color):
    np_img = np.array(img)
    mask = (np_img != crop_color)[..., 0]   # compute a mask
    x0, x1 = get_first_last(mask, 0)        # find boundaries along x axis
    y0, y1 = get_first_last(mask, 1)        # find boundaries along y axis
    return img.crop((x0, y0, x1, y1)) 

def result_to_dataframe(result: dict) -> pd.DataFrame:
    df = pd.DataFrame(
        {
            "解盤內容": result
        }
    )
    df.index.name = "技術分析工具"
    return df
def display_result( result) :
    result_df = result_to_dataframe( result)
    result_md   = result_df.to_markdown(tablefmt="grid")
    result_html = markdown.markdown(result_md, extensions=['markdown_grid_tables:GridTableExtension'])
    display(HTML(result_html))

In [3]:
# =========================
# PROMPT BUILDERS
# =========================

def build_system_prompt(instrument: dict) -> str:
    return f"""
你是一位技術分析師，使用「看圖優先，數據輔助」。

【總原則】
- 以圖形觀察為主，數據僅作輔助確認
- 最終輸出必須是自然中文，不可像資料表或工程欄位
- 不可暴露內部分析流程、視窗、欄位名稱或判斷規格
- 型態可以不存在，不可硬湊
- 各分析欄位必須依自身規則獨立完成判斷
- 只有「整體評價」可以整合其他欄位結論

【欄位獨立判斷原則（強制）】
以下欄位必須各自獨立完成判斷，不可互相引用、互相驗證、互相補位、或為了保持一致而改寫內容：
- K棒
- K線圖
- 成交量
- 型態
- 移動平均線
- MACD指標
- KD指標

各欄位只能依自身分析維度與規則輸出結論：
- K棒只看K棒表現
- K線圖只看價格走勢節奏與位置
- 成交量只看量價關係
- 型態只看價格結構
- 移動平均線只看均線方向與支撐壓力
- MACD指標只看MACD訊號
- KD指標只看KD訊號

禁止以下行為：
- 不可用型態結論改寫K線圖
- 不可用K線圖語氣補充型態
- 不可用成交量、均線、MACD、KD去支持或否定型態
- 不可因其他欄位偏多、偏空或整理，而回頭修改本欄位結論
- 不可為了整體一致性而強行統一各欄位說法

【整體評價整合原則（強制）】
只有「整體評價」欄位可以整合前述所有欄位的結論。
整體評價必須：
- 綜合K棒、K線圖、成交量、型態、移動平均線、MACD指標、KD指標
- 在訊號一致時給出較明確結論
- 在訊號分歧時明確指出分歧來源
- 統整出當前偏多、偏空或整理的整體判讀，以及關鍵支撐、壓力與風險

【MACD輸出規範】
只允許：DIF線、MACD線、OSC柱體
禁止：macd / macdsignal / macdhist / 訊號線

【標的資訊】
{json.dumps(instrument, ensure_ascii=False)}
""".strip()


def build_phase1_prompt(stock_id: str) -> str:
    return f"""
你會收到三張圖：
1. 長週期（240）
2. 中週期（120）
3. 短週期（60）

--------------------------------------------------

【Phase 1 任務】
此階段只負責多週期「型態結構辨識」，輸出 structure JSON。
重點是辨識各週期是否存在明確價格型態，並提供該型態的時間範圍與簡要結構說明。

--------------------------------------------------

【型態判斷順序（強制）】
型態辨識必須依以下順序進行：
1. 短週期
2. 中週期
3. 長週期

每個週期必須先獨立完成判斷，再進入下一週期，不可跳過或反向推論。

--------------------------------------------------

【可見範圍原則（強制）】
每個週期的型態判斷，只能依該週期圖上「實際可見」的價格結構進行。

- 若某型態在該週期中無法完整觀察（例如缺少關鍵低點、高點或頸線），不得判定為該型態
- 不可引用其他週期才看得到的結構，作為本週期型態判斷依據

--------------------------------------------------

【跨週期影響限制（強制）】
各週期型態判斷必須相互獨立，禁止以下行為：

- 不可因長週期存在某型態，就要求中週期或短週期套用相同型態
- 不可為了讓結構一致，而修改本週期已成立的型態判斷
- 不可由長週期反向決定短週期型態

--------------------------------------------------

【型態一致性條件（限制性允許）】
只有在以下條件同時成立時，跨週期型態才可一致：

- 較大週期的型態，在較小週期中也能清楚觀察到對應結構
- 小週期走勢明確屬於該型態的延伸段（例如突破後拉回、回測不破等）

若不符合上述條件，各週期應維持不同型態判斷。

--------------------------------------------------

【型態判斷資料來源（強制）】
型態判斷僅能依據價格結構進行，包括：
- 高點與低點的相對關係
- 是否出現明確支撐與壓力區
- 是否形成典型價格結構（雙重底、雙重頂、頭肩底、頭肩頂、上升三角形、下降三角形、對稱三角形、箱型整理、旗形等）

禁止使用以下資訊作為型態判斷依據：
- KD指標
- MACD指標
- 成交量
- 均線
- 趨勢語言（多頭、空頭、整理）作為型態替代

--------------------------------------------------

【型態與趨勢分離（強制）】
型態判斷僅負責辨識「是否存在明確技術型態」。

- 若無法辨識明確型態，structure_type 應維持為 none
- 不可將趨勢（多頭、空頭、整理）作為型態的替代
- 不可因沒有型態，就改寫成高檔整理、區間盤整、狹幅整理、多頭整理、偏空震盪等描述

--------------------------------------------------

【最近型態優先】
若同一週期可辨識多個型態，只選擇：
- 最近
- 仍影響當前價格
- 結構最完整
的那一個作為主型態。

--------------------------------------------------

【區間不得覆蓋底型（條件成立時）】
若在同一週期內，明確可見：
- 兩個相近低點
- 中間反彈區
- 第二低點未破第一低點
- 價格重新走高

則應使用 double_bottom，不可只因右側仍在震盪，就直接改判為 box。

但前提是：該雙重底必須在該週期圖上可完整看見。
若該週期圖上看不完整，則不得強行判為 double_bottom。

--------------------------------------------------

【型態輸出規則】
- structure_type：主型態或 none（僅內部使用）
- structure_range：型態時間；若無明確型態則填 none
- note：僅描述價格結構本身，不可混入指標、均線、成交量、趨勢結論

--------------------------------------------------

【型態描述限制（強制）】
在 note 中，禁止出現以下內容：
- KD相關用語（K線、D線、KD雙線、超買、超賣、鈍化等）
- MACD相關用語（DIF線、MACD線、OSC柱體、零軸等）
- 均線描述（5日線、10日線、20日線、60日線等）
- 成交量描述（量縮、量增、背離、換手等）
- 趨勢補位語言（偏多、偏空、整理、多頭、空頭）

note 必須只描述價格結構，例如：
- 兩次回測低點接近，中間有反彈，之後重新走高
- 在固定區間內來回震盪，上下界清楚
- 高點逐步下移、低點逐步墊高，形成收斂

--------------------------------------------------

【支撐壓力】
來源僅限：
- 高低點
- 區間上緣 / 下緣
- 結構頸線 / 關鍵轉折區

此處僅作為輔助辨識型態，不可延伸成均線、指標或趨勢分析。

--------------------------------------------------

【need_price_data】
只有以下情況才回傳 YES：
- 關鍵價位在圖上無法明確辨識
- 型態時間範圍無法穩定確認
- 圖上品質不足，需數據輔助確認價格位置

不得因 KD / MACD / 成交量 / 均線而要求 price_data。

--------------------------------------------------

【輸出 JSON】
{{
  "stock_id": "{stock_id}",
  "long_view": {{
    "structure_type": "...",
    "structure_range": "...",
    "note": "..."
  }},
  "mid_view": {{
    "structure_type": "...",
    "structure_range": "...",
    "note": "..."
  }},
  "short_view": {{
    "structure_type": "...",
    "structure_range": "...",
    "note": "..."
  }},
  "need_price_data": "YES | NO",
  "need_price_data_reason": ""
}}
""".strip()


def build_phase2_prompt(
    stock_id: str,
    structure: dict,
    instrument: dict,
    price_data: Optional[dict] = None
) -> str:
    prompt = f"""
請分析 {stock_id}。

【結構】
{json.dumps(structure, ensure_ascii=False)}

【標的資訊】
{json.dumps(instrument, ensure_ascii=False)}

--------------------------------------------------

【本階段任務】
你必須輸出以下八個欄位：
- K棒
- K線圖
- 成交量
- 型態
- 移動平均線
- MACD指標
- KD指標
- 整體評價

其中前七個欄位必須各自獨立判斷，只有「整體評價」可以整合前七個欄位。

--------------------------------------------------

【欄位獨立判斷原則（強制）】
以下欄位必須各自獨立完成判斷，不可互相引用、互相驗證、互相補位、或為了保持一致而改寫內容：
- K棒
- K線圖
- 成交量
- 型態
- 移動平均線
- MACD指標
- KD指標

在生成任一欄位時：
- 不可讀取或引用其他欄位已生成的內容
- 每個欄位必須視為獨立任務，僅依自身規則完成
- 不可為了整體一致性而強行統一各欄位說法

--------------------------------------------------

【整體評價整合原則（強制）】
只有「整體評價」欄位可以整合前述所有欄位的結論。

整體評價必須：
- 綜合K棒、K線圖、成交量、型態、移動平均線、MACD指標、KD指標
- 在訊號一致時給出較明確結論
- 在訊號分歧時明確指出分歧來源
- 統整出當前偏多、偏空或整理的整體判讀，以及關鍵支撐、壓力與風險

--------------------------------------------------

【型態欄位規則（強制）】

一、型態的唯一來源：
- 「型態」欄位只能依 structure 判斷
- structure 是型態欄位的唯一來源，不是參考來源

二、型態名稱必須標準化：
- double_bottom → 雙重底
- double_top → 雙重頂
- head_and_shoulders_bottom → 頭肩底
- head_and_shoulders_top → 頭肩頂
- ascending_triangle → 上升三角形
- descending_triangle → 下降三角形
- symmetrical_triangle → 對稱三角形
- box → 箱型整理
- flag → 旗形

禁止：
- 雙重底 → 雙底 / W底
- 箱型整理 → 其他名稱
- 上升三角形 → 收斂三角 / 三角整理
- 旗形 → 小旗 / 整理旗

三、型態欄位只能有一個主型態：
- 不可並列兩個型態
- 不可用次階型態覆蓋主型態

四、若 structure_type = none：
- 「型態」欄位不可輸出 none
- 不可寫「沒有型態」
- 不可用趨勢、整理、盤整等語句補位
- 此時請改寫成中性且誠實的自然語句，只說目前未見明確技術型態，避免補入趨勢結論

五、型態描述只能保留：
- 型態名稱
- 型態日期（若有）
- 當前狀態

六、型態描述禁止出現：
- K線圖用語（高檔整理、震盪節奏、推升、拉回等）
- 成交量用語（量縮、量增、背離等）
- 均線用語（5日線、20日線、均線帶等）
- KD用語（K線、D線、超買、超賣、鈍化等）
- MACD用語（DIF線、MACD線、OSC柱體、零軸等）
- 趨勢語言（偏多、偏空、多頭、空頭、整理）作為型態替代

--------------------------------------------------

【K線圖欄位規則（強制）】
K線圖欄位只描述：
- 價格走勢節奏
- 當前位置
- 支撐壓力之間的互動
- 接下來較可能的節奏

K線圖禁止：
- 引用型態名稱（雙重底、雙重頂、箱型整理等）
- 引用均線、成交量、MACD、KD結論
- 用型態來解釋走勢
- 反向影響型態欄位

--------------------------------------------------

【K棒欄位規則（強制）】
K棒只代表近期K棒節奏與市場反應。
只可描述：
- 小紅小黑交錯
- 連紅 / 連黑
- 長紅 / 長黑
- 上下影線
- 吞噬、轉折、承接、追價降溫等近期反應

禁止：
- 引用型態名稱
- 引用均線 / 成交量 / MACD / KD
- 使用長中短週期語言

--------------------------------------------------

【成交量欄位規則（強制）】
成交量只負責量價關係。
優先使用：
- 價漲量增
- 價跌量縮
- 量縮整理

輔助：
- 量能放大 / 縮小
- 換手
- 價量背離

禁止：
- 用成交量去支持或否定型態
- 引用均線、MACD、KD結論
- 引用型態名稱

【成交量時間尺度規則（強制）】
成交量只允許使用短週期（60根）作為內部判讀依據，但不可在最終輸出中寫出 60根。

若出現價量背離：
- 必須寫在「成交量」
- 並在「整體評價」提高權重

--------------------------------------------------

【移動平均線欄位規則（強制）】
均線分析只負責：
- 5日線、10日線、20日線、60日線的位置與方向
- 哪條均線提供支撐或壓力
- 均線是否糾結、翻揚、下彎

禁止：
- 引用型態名稱
- 用均線去支持或否定型態
- 引用MACD / KD / 成交量結論

【均線定義】
短期均線：
- 5日線
- 10日線

中期均線：
- 20日線
- 60日線

【均線趨勢判斷（強制）】
- 5日線 + 10日線 同時上揚 → 短線偏多
- 5日線 + 10日線 同時下彎 → 短線偏空
- 其他 → 短線盤整

- 20日線 + 60日線 同時上揚 → 中期偏多
- 20日線 + 60日線 同時下彎 → 中期偏空
- 其他 → 中期盤整

【均線輸出規則（強制）】
必須明確指出：
- 哪些均線提供支撐或壓力
- 對應哪些價位

禁止：
- 只寫「均線支撐」
- 只寫「均線帶」
- 不說明是哪條均線

--------------------------------------------------

【MACD指標欄位規則（強制）】
MACD指標只負責：
- DIF線
- MACD線
- OSC柱體
- 零軸位置
- 動能增強 / 減弱
- 柱體擴大 / 縮小
- 背離

禁止：
- 用MACD去支持或否定型態
- 引用K線圖、均線、KD、成交量結論
- 使用訊號線、macd、macdsignal、macdhist等詞

--------------------------------------------------

【KD指標欄位規則（強制）】
KD指標只負責：
- K線
- D線
- KD雙線
- 高檔 / 低檔
- 超買區 / 超賣區
- 黃金交叉 / 死亡交叉
- 回升 / 回落
- 鈍化

禁止：
- 用KD去支持或否定型態
- 引用K線圖、均線、MACD、成交量結論
- 使用偏強區 / 偏弱區 / 強勢區 / 弱勢區 / 動能區等詞

【KD主詞規則（強制）】
- 不要只寫「KD...」
- 必須明確使用 K線 / D線 / KD雙線

--------------------------------------------------

【日期格式規範（強制）】
日期必須使用：
- M月D日（不可補零）

例如：
- 3月7日
- 11月21日
- 2025年11月21日至2026年3月20日

--------------------------------------------------

【標的代號不可出現在輸出文字中（強制）】
最終輸出的各欄位內容不可直接出現：
- 股票代號
- 指數代號
- ETF代號
- 英文代碼
- 標的名稱作為句首主詞

【禁止寫法】
- TAIEX目前屬...
- 2330目前...
- 0050目前...
- 本標的目前...

【正確寫法】
- 目前走勢屬於...
- 現階段結構偏向...
- 整體來看屬於...
- 當前盤勢仍是...

--------------------------------------------------

【內部規格不可外顯（強制）】
以下內容屬於內部分析依據，不可直接出現在最終輸出文字中：
- 3~5天
- 60根
- 長週期 / 中週期 / 短週期
- 240 / 120 / 60
- 結構判斷流程
- 指標判讀門檻
- 任何固定視窗分析說法
- Phase 1 / Phase 2
- structure_type / long_view / mid_view / short_view / structure_range

--------------------------------------------------

【數據使用規範（強制）】
可以用數據判斷，但：
- 不可提及資料來源
- 不可用欄位名稱
- 不可像資料表

關鍵數字可輸出，但必須加單位。
請依 instrument 中的單位資訊輸出：
- 價格 / 指數點位 → 使用 price_unit
- 成交量 → 使用 volume_unit

【禁止】
- 只寫數字不帶單位
- 同一份輸出中單位前後不一致
- 使用原始資料格式，例如：
  - K=84.91、D=77.51、Volume=123456
  - macd=1.23

--------------------------------------------------

【輸出 JSON】
{{
  "K棒": "...",
  "K線圖": "...",
  "成交量": "...",
  "型態": "...",
  "移動平均線": "...",
  "MACD指標": "...",
  "KD指標": "...",
  "整體評價": "..."
}}
""".strip()

    if price_data:
        prompt += "\n\n【補充數據】\n" + json.dumps(price_data, ensure_ascii=False)

    return prompt


# =========================
# AGENT
# =========================

class StockAnalysisAgent:
    def __init__(self, model: str = "gpt-5.2", sqlite_db_path = 'data/stock.db', debug = False):
        # API Key不明寫，從環境變數中取得
        load_dotenv(find_dotenv())
        api_key = os.environ.get('OPENAI_API_KEY_TOKEN')
        self.client = OpenAI(api_key=api_key)
        self.model = model
        # 我的內部屬性
        ### 設定除錯旗標 ###
        self._debug = debug
        ### 設定股票代號 ###
        self._stock_id = None
        # 我的初始化程序
        ### 連線資料庫 ###
        if os.path.isfile(sqlite_db_path):
            self._conn = sqlite3.connect( sqlite_db_path)
        else :
            raise ValueError('The database file does not exist.')
        ### 從資料庫中載入「台股總覽 TaiwanStockInfo」 ###
        try :
            self._df_stock_info = pd.read_sql("SELECT * FROM StockInfo", self._conn)
        except Exception as e:
            raise ValueError('An error occurred while loading TaiwanStockInfo.')

        self.lookbacks = {
            "long": 240,
            "mid": 120,
            "short": 60,
        }

    # 我的內部方法
    ### 列印除錯訊息之內部方法 ###
    def _debug_print( self, msg) :
        if self._debug is True :
            print("ＤＥＢＵＧ ： {}".format(msg))
    ### 重置內部屬性之內部方法 ###
    def _reset_internal_attribute( self) :
        self._daily_price_df        = None
        self._sma_df                = None
        self._kd_df                 = None
        self._macd_df               = None
        self._instrument_type       = None
        self._price_unit            = None
        self._volume_unit           = None
        ### 從資料庫中載入日Ｋ資料之內部方法 ###
    def _loading_price_data( self, stock_id) :
        # 將載入的「台股總覽 TaiwanStockInfo」進行格式轉換
        df_stock_info = self._df_stock_info.set_index(self._df_stock_info['StockID'],inplace=False)
        df_stock_info = df_stock_info.drop(columns=['StockID'])
        
        # 判斷股票代碼(stock_id)是否存在於「台股總覽 TaiwanStockInfo」中
        if stock_id in df_stock_info.index:
            
            # 取得該股票代碼的產業分類
            individual_stock_info = df_stock_info.loc[stock_id]
            industry_category     = individual_stock_info['IndustryCategory']
            
            # 設定開始日期與結束日期
            current_date       = datetime.datetime.today()
            daily_end_date     = current_date.strftime('%Y-%m-%d')
            daily_start_date,_ = get_monday_to_sunday((current_date - datetime.timedelta(days=730)).strftime('%Y-%m-%d'))
            
            # 讀取日Ｋ價格資料
            sql_cmd = "SELECT * FROM DailyPrice WHERE StockID='{}' AND (Date BETWEEN '{}' AND '{}') ORDER BY Date".format(stock_id,daily_start_date,daily_end_date)
            try :
                daily_price_df = pd.read_sql( sql_cmd, self._conn)
            except Exception as e:
                self._debug_print('讀取日Ｋ資料錯誤，錯誤訊息＝ {}'.format(str(e)))
                return False
            # 調整停牌或未交易之日Ｋ價格資料
            correcting_zero_price_issue( daily_price_df, debug = self._debug)
            # 格式轉換：日期格式、成交量(成交值)
            daily_price_df           = daily_price_df.drop(columns=['SerialNo','StockID'])
            daily_price_df['Date']   = daily_price_df['Date'].astype('datetime64[ns]')
            daily_price_df.set_index(daily_price_df['Date'],inplace=True)
            daily_price_df           = daily_price_df.drop(columns=['Date'])
            if industry_category == 'Index' or industry_category == '大盤' :
                daily_price_df           = daily_price_df.drop(columns=['Volume'])
                daily_price_df           = daily_price_df.rename(columns={'Value':'Volume'})
                daily_price_df['Volume'] = daily_price_df['Volume'].div(100000000.00)
                daily_price_df['Volume'] = daily_price_df['Volume'].round(2)
            else :
                daily_price_df           = daily_price_df.drop(columns=['Value'])
                daily_price_df['Volume'] = daily_price_df['Volume'].div(1000)
                daily_price_df['Volume'] = daily_price_df['Volume'].round()
                daily_price_df['Volume'] = daily_price_df['Volume'].astype('int64')
            self._daily_price_df         = daily_price_df
                
            if industry_category == 'Index' or industry_category == '大盤' :
                self._instrument_type       = 'index'
                self._price_unit            = '點'
                self._volume_unit           = '億元'
            else :
                self._instrument_type       = 'stock'
                self._price_unit            = '元'
                self._volume_unit           = '張'
                
            return True
        return False
    ### 使用talib程式庫計算技術指標之內部方法 ###
    def _technical_indicators( self) :
        # 日Ｋ價格資料轉換為talib格式
        daily_price_df_talib          = self._daily_price_df.copy()
        daily_price_df_talib.columns  = [ i.lower() for i in daily_price_df_talib.columns]
        
        # 計算移動平均線
        talib_sma5        = SMA( daily_price_df_talib, timeperiod=5)
        talib_sma10       = SMA( daily_price_df_talib, timeperiod=10)
        talib_sma20       = SMA( daily_price_df_talib, timeperiod=20)
        talib_sma60       = SMA( daily_price_df_talib, timeperiod=60)
        talib_sma120      = SMA( daily_price_df_talib, timeperiod=120)
        talib_sma240      = SMA( daily_price_df_talib, timeperiod=240)
        # 設定名稱
        talib_sma5.name   = 'SMA5'
        talib_sma10.name  = 'SMA10'
        talib_sma20.name  = 'SMA20'
        talib_sma60.name  = 'SMA60'
        talib_sma120.name = 'SMA120'
        talib_sma240.name = 'SMA240'
        # 合併各條均線
        talib_sma_df      = pd.concat([talib_sma5, talib_sma10, talib_sma20, talib_sma60, talib_sma120, talib_sma240], axis=1)
        # 取小數點後兩位
        self._sma_df      = talib_sma_df.round(2)
        
        # 計算ＫＤ指標
        talib_daily_kd = STOCH( daily_price_df_talib, fastk_period=9, slowk_period=3, slowd_period=3)
        # 取小數點後兩位
        self._kd_df    = talib_daily_kd.round(2)
        
        # 計算ＭＡＣＤ指標
        talib_daily_macd = MACD( daily_price_df_talib, fastperiod=12, slowperiod=26, signalperiod=9)
        # 取小數點後兩位
        self._macd_df    = talib_daily_macd.round(2)
    ### 載入價量資料並計算技術指標 ###
    def _loading_price_data_and_technical_indicators( self, stock_id) :
        if self._stock_id is None or self._stock_id != stock_id :
            self._debug_print('設置或重置內部屬性')
            self._reset_internal_attribute()
            self._debug_print('載入價量資料,股票代碼 ＝ {}'.format(stock_id))
            if self._loading_price_data( stock_id) is True :
                self._debug_print('計算技術指標')
                self._technical_indicators()
                self._stock_id = stock_id
            else :
                raise ValueError("讀取價量資料錯誤！")


    # ===== TODO =====

    def classify_instrument(self, stock_id: str) -> Dict[str, str]:
        
        self._loading_price_data_and_technical_indicators( stock_id)
        
        return {"instrument_type":self._instrument_type , "price_unit":self._price_unit , "volume_unit": self._volume_unit}

    def get_analysis_dates(self, prev_days: int) -> Tuple[str, str]:

        if  self._daily_price_df is not None :
            end_date   = self._daily_price_df.iloc[-1].name.strftime('%Y-%m-%d')
            start_date = self._daily_price_df.iloc[-1-prev_days].name.strftime('%Y-%m-%d')
            self._debug_print('開始日期 ＝ {} （輸入參數 ＝ {} ）， 結束日期 ＝ {} '.format(start_date,prev_days,end_date))
        else :
            raise ValueError("沒有價量資料！")
        
        return (start_date,end_date)


    def get_chart(self, stock_id: str, start_date: str, end_date: str) -> str:
        
        self._loading_price_data_and_technical_indicators( stock_id)
        
        range_prices = self._daily_price_df[start_date:end_date]
        range_sma    = self._sma_df[start_date:end_date]
        range_kd     = self._kd_df[start_date:end_date]
        range_macd   = self._macd_df[start_date:end_date]
        
        # 設定K線格式
        mc = mpf.make_marketcolors(up='xkcd:light red', down='xkcd:almost black', inherit=True)
        s  = mpf.make_mpf_style(base_mpf_style='yahoo', marketcolors=mc)
        
        # ＫＤ指標：超買線
        kd_overbought_line = [80] * range_prices.shape[0]
        # ＫＤ指標：賣超線
        kd_oversold_line   = [20] * range_prices.shape[0]

        # 設定技術指標
        added_plots = [
            mpf.make_addplot(range_sma['SMA5'],width=0.8,color='xkcd:red brown'),
            mpf.make_addplot(range_sma['SMA10'],width=0.8,color='xkcd:dark sky blue'),
            mpf.make_addplot(range_sma['SMA20'],width=0.8,color='xkcd:violet'),
            mpf.make_addplot(range_sma['SMA60'],width=0.8,color='xkcd:orange'),
            mpf.make_addplot(range_kd['slowk'],width=0.8,panel=2,secondary_y=False,color='xkcd:red',ylabel='KD'),
            mpf.make_addplot(range_kd['slowd'],width=0.8,panel=2,secondary_y=False,color='xkcd:blue'),
            mpf.make_addplot(kd_overbought_line,width=0.8,panel=2,secondary_y=False,linestyle='--',color='xkcd:green'),
            mpf.make_addplot(kd_oversold_line,width=0.8,panel=2,secondary_y=False,linestyle='--',color='xkcd:green'),
            mpf.make_addplot(range_macd['macdhist'],type='bar',panel=3,secondary_y=False,color='xkcd:grey',ylabel='MACD'),
            mpf.make_addplot(range_macd['macd'],width=0.8,panel=3,secondary_y=False,color='xkcd:red'),
            mpf.make_addplot(range_macd['macdsignal'],width=0.8,panel=3,secondary_y=False,color='xkcd:blue')
        ]

        # 設定Ｋ線圖X軸座標值
        ticks        = []
        tlabs        = []
        label_count  = 0
        total_k_line = range_prices.shape[0]
        for idx in range(total_k_line) :
            timestamp = range_prices.iloc[idx].name
            if idx == 0 :
                ticks.append(idx)
                tlabs.append(timestamp.strftime('%Y-%m-%d'))
                label_count = 0
            elif (idx+1) == total_k_line:
                if label_count < 5 :
                    # 移除上一筆資料
                    ticks.pop()
                    tlabs.pop()
                ticks.append(idx)
                tlabs.append(timestamp.strftime('%Y-%m-%d'))
            elif label_count > 10 :
                ticks.append(idx)
                tlabs.append(timestamp.strftime('%Y-%m-%d'))
                label_count = 0
            label_count += 1
        
        # 繪製Ｋ線圖
        tmp = io.BytesIO()
        kwargs = dict(type='candle', style=s, figratio=(16,9), volume=True, addplot=added_plots, main_panel=0, volume_panel=1, panel_ratios=(5, 1, 2, 2), num_panels=4, datetime_format='%Y-%m-%d', returnfig=True, savefig=dict(fname=tmp))
        fig, axlist = mpf.plot(range_prices,**kwargs)
        axlist[-2].set_xticks(ticks,labels=tlabs,ha='right')
        tmp = None
        
        # 保存Ｋ線圖
        img = io.BytesIO()
        fig.savefig(img,format='png')

        # 刪除空白
        image    = Image.open(img).convert("RGB")
        image    = crop_borders(image, crop_color=(255, 255, 255))
        crop_img = io.BytesIO()
        image.save(crop_img,format='PNG')
        
        # 將影像檔(PNG)編碼為Base64
        base64_image = base64.b64encode(crop_img.getvalue()).decode()
        
        return base64_image

    def get_price_data(self, stock_id: str, start_date: str, end_date: str) -> Dict[str, Any]:
        
        self._loading_price_data_and_technical_indicators( stock_id)
        
        range_prices     = self._daily_price_df[start_date:end_date]
        range_sma        = self._sma_df[start_date:end_date]
        range_kd         = self._kd_df[start_date:end_date]
        range_macd       = self._macd_df[start_date:end_date]
        
        ret_records_list = []
        
        for idx in range(len(range_prices)) :
            record = {
                "date": range_prices.iloc[idx].name.strftime('%Y-%m-%d'),
                "open": range_prices.iloc[idx]['Open'],
                "high": range_prices.iloc[idx]['High'],
                "low": range_prices.iloc[idx]['Low'],
                "close": range_prices.iloc[idx]['Close'],
                "volume": range_prices.iloc[idx]['Volume'],
                "sma5": range_sma.iloc[idx]['SMA5'],
                "sma10": range_sma.iloc[idx]['SMA10'],
                "sma20": range_sma.iloc[idx]['SMA20'],
                "sma60": range_sma.iloc[idx]['SMA60'],
                "macd": range_macd.iloc[idx]['macd'],
                "macdsignal": range_macd.iloc[idx]['macdsignal'],
                "macdhist": range_macd.iloc[idx]['macdhist'],
                "slowk": range_kd.iloc[idx]['slowk'],
                "slowd": range_kd.iloc[idx]['slowd'] 
            }
            ret_records_list.append(record)
        
        summary = {'latest_close' : range_prices.iloc[-1]['Close']}
            
        return {"records":ret_records_list,"summary":summary}

    # ===== internal =====

    def _image_url(self, b64: str) -> str:
        return f"data:image/png;base64,{b64}"

    def _safe_json(self, text: str) -> dict:
        text = text.strip()
        if text.startswith("```"):
            lines = text.splitlines()
            if lines and lines[0].startswith("```"):
                lines = lines[1:]
            if lines and lines[-1].startswith("```"):
                lines = lines[:-1]
            text = "\n".join(lines).strip()
            if text.lower().startswith("json"):
                text = text[4:].strip()
        return json.loads(text)

    def _call(self, system: str, user: str, images: Dict[str, str]) -> str:
        content = [{"type": "input_text", "text": user}]
        for k in ["long", "mid", "short"]:
            content.append({"type": "input_image", "image_url": images[k]})

        r = self.client.responses.create(
            model=self.model,
            input=[
                {"role": "system", "content": system},
                {"role": "user", "content": content}
            ]
        )
        return r.output_text

    def _get_charts(self, stock_id: str) -> dict:
        result = {}
        for k, days in self.lookbacks.items():
            s, e = self.get_analysis_dates(days)
            b64 = self.get_chart(stock_id, s, e)
            result[k] = self._image_url(b64)
        return result

    # ===== Phase 1 =====

    def extract_structure(self, stock_id: str) -> dict:
        instrument = self.classify_instrument(stock_id)
        charts = self._get_charts(stock_id)

        raw = self._call(
            build_system_prompt(instrument),
            build_phase1_prompt(stock_id),
            charts
        )
        return self._safe_json(raw)

    # ===== Phase 2 =====

    def generate_analysis(self, stock_id: str, structure: dict) -> dict:
        instrument = self.classify_instrument(stock_id)
        charts = self._get_charts(stock_id)

        price_data = None
        if structure.get("need_price_data") == "YES":
            s, e = self.get_analysis_dates(120)
            price_data = self.get_price_data(stock_id, s, e)

        raw = self._call(
            build_system_prompt(instrument),
            build_phase2_prompt(stock_id, structure, instrument, price_data),
            charts
        )
        return self._safe_json(raw)

    # ===== API =====

    def run(self, stock_id: str, debug: bool = False) -> dict:
        structure = self.extract_structure(stock_id)
        analysis = self.generate_analysis(stock_id, structure)

        if debug:
            return {
                "結構": structure,
                "分析": analysis
            }
        return analysis

In [4]:
agent = StockAnalysisAgent(debug=True)

In [5]:
# 測試：加權指數
result = agent.run('TAIEX',debug=True)
print('除錯資訊 ＝  {}'.format(result))
display_result(result['分析'])

ＤＥＢＵＧ ： 設置或重置內部屬性
ＤＥＢＵＧ ： 載入價量資料,股票代碼 ＝ TAIEX
ＤＥＢＵＧ ： 計算技術指標
ＤＥＢＵＧ ： 開始日期 ＝ 2025-03-25 （輸入參數 ＝ 240 ）， 結束日期 ＝ 2026-03-20 
ＤＥＢＵＧ ： 開始日期 ＝ 2025-09-15 （輸入參數 ＝ 120 ）， 結束日期 ＝ 2026-03-20 
ＤＥＢＵＧ ： 開始日期 ＝ 2025-12-12 （輸入參數 ＝ 60 ）， 結束日期 ＝ 2026-03-20 
ＤＥＢＵＧ ： 開始日期 ＝ 2025-03-25 （輸入參數 ＝ 240 ）， 結束日期 ＝ 2026-03-20 
ＤＥＢＵＧ ： 開始日期 ＝ 2025-09-15 （輸入參數 ＝ 120 ）， 結束日期 ＝ 2026-03-20 
ＤＥＢＵＧ ： 開始日期 ＝ 2025-12-12 （輸入參數 ＝ 60 ）， 結束日期 ＝ 2026-03-20 
除錯資訊 ＝  {'結構': {'stock_id': 'TAIEX', 'long_view': {'structure_type': 'none', 'structure_range': 'none', 'note': '圖中走勢以一段延伸上行為主，途中雖有回落與再攻，但缺少可明確框定的頸線、上下邊界或對稱/收斂的高低點序列，因此無法判定為典型雙底、頭肩、三角形或箱型等結構。'}, 'mid_view': {'structure_type': 'none', 'structure_range': 'none', 'note': '可見範圍內先上行後出現一段回落，隨後急拉創高再回撤，右側轉為較窄幅的往返，但上下界與轉折點數量不足以形成清楚的箱型或三角收斂；同時也看不到足夠完整的雙底/頭肩所需關鍵轉折與頸線結構。'}, 'short_view': {'structure_type': 'box', 'structure_range': '2026-03-07 ~ 2026-03-20', 'note': '高點多次受限於約34500附近，低點多次回測約33500附近後回到區間內，形成上下邊界相對清楚的往返震盪結構。'}, 'need_price_data': 'NO', 'need_price_data_reason': ''}, '分析': 

技術分析工具,解盤內容
K棒,近期K棒以小紅小黑交錯為主，連續性不強，顯示追價與承接都偏保守。期間可見數次帶上下影線的K棒，代表盤中拉抬與回吐都存在；整體呈現「拉不太上去、也不易一路跌下來」的反覆測試節奏。
K線圖,價格在前段明顯走高後，右側進入高位區間內的往返，重心大致落在3萬3千多點到3萬4千多點之間來回。上方接近3萬4千5百點附近時較常遇到推進放慢的情況；回到3萬3千5百點附近則較容易出現回穩。接下來較可能延續區間內反覆試探：若上緣被有效越過，才有機會轉回向上延伸；若下緣失守，則容易引發一段較明顯的回撤。
成交量,成交量在高位區間內屬於時有放大、但未形成連續放量的狀態，較常見「價格上去量能跟進有限、回落時量能也未明顯失控」的型態。整體偏向量縮整理的量價關係，顯示短線資金以觀望與短打居多，尚未看到明確的趨勢性量能接力。
型態,箱型整理（3月7日至3月20日）；目前仍在箱型範圍內運行，上緣約34500點、下緣約33500點，尚未出現明確脫離箱型的狀態。
移動平均線,5日線與10日線在右側走向趨於走平、彼此距離不大，短線屬於盤整；價格多在兩條短期均線附近來回，顯示支撐與壓力反覆切換。20日線仍在上方區段附近提供支撐參考，位置約落在3萬3千多點中段；60日線維持上揚並位於更下方，約在3萬2千點附近，作為較後方的支撐帶。上方壓力則落在短期均線上方的3萬4千多點區域，需看到價格能站穩其上，才算減輕短線反覆拉回的壓力。
MACD指標,DIF線與MACD線位於零軸上方但呈現回落後趨平，顯示上行動能先前轉弱、目前在高位鈍化。OSC柱體多在零軸附近縮小、正負來回切換，代表動能缺乏連續性，短線容易出現來回震盪而非單邊擴張。
KD指標,K線與D線自低檔回升後向上推進，KD雙線目前位於中高檔區域，短線偏向回升結構。由於已接近上方區間，後續若K線在高檔出現轉折回落、並跌破D線，則容易再度拉回；若K線能在高檔維持並帶動D線上行，則有機會延續高檔震盪偏上的表現。
整體評價,整體來看偏向「高位整理」。價格行為呈現區間往返，上方約34500點附近壓力明顯、下方約33500點附近支撐反覆出現；型態上也明確落在箱型整理內。均線面中期（20日線、60日線）仍維持向上，代表下檔結構尚有支撐，但短期（5日線、10日線）走平使得上攻續航不足。指標面MACD動能趨於鈍化、OSC柱體在零軸附近縮小，與KD回升的訊號略有分歧：KD偏向反彈延續，但MACD顯示推升力道不強，容易走成「反彈後再回到區間」的節奏。操作上關鍵在箱型邊界：站穩34500點之上較有利延伸上攻；若跌破33500點，回撤風險會明顯提高。


In [6]:
# 測試：宏碁(2353)
result = agent.run('2353',debug=True)
print('除錯資訊 ＝  {}'.format(result))
display_result(result['分析'])

ＤＥＢＵＧ ： 設置或重置內部屬性
ＤＥＢＵＧ ： 載入價量資料,股票代碼 ＝ 2353
ＤＥＢＵＧ ： 計算技術指標
ＤＥＢＵＧ ： 開始日期 ＝ 2025-03-25 （輸入參數 ＝ 240 ）， 結束日期 ＝ 2026-03-20 
ＤＥＢＵＧ ： 開始日期 ＝ 2025-09-15 （輸入參數 ＝ 120 ）， 結束日期 ＝ 2026-03-20 
ＤＥＢＵＧ ： 開始日期 ＝ 2025-12-12 （輸入參數 ＝ 60 ）， 結束日期 ＝ 2026-03-20 
ＤＥＢＵＧ ： 開始日期 ＝ 2025-03-25 （輸入參數 ＝ 240 ）， 結束日期 ＝ 2026-03-20 
ＤＥＢＵＧ ： 開始日期 ＝ 2025-09-15 （輸入參數 ＝ 120 ）， 結束日期 ＝ 2026-03-20 
ＤＥＢＵＧ ： 開始日期 ＝ 2025-12-12 （輸入參數 ＝ 60 ）， 結束日期 ＝ 2026-03-20 
除錯資訊 ＝  {'結構': {'stock_id': '2353', 'long_view': {'structure_type': 'none', 'structure_range': 'none', 'note': '可見區間內高點一路走低、低點也持續下移，中後段出現一段在相近價位附近反覆測試的低點區，但缺少清楚的頸線/上緣可用來界定為完整的反轉或整理型態。'}, 'mid_view': {'structure_type': 'none', 'structure_range': 'none', 'note': '前段由高點逐步下移並延伸到低點區，之後進入反覆震盪；右側雖有上推，但上下界與關鍵頸線不夠明確，無法在圖上完整界定為箱型、三角形或頭肩等典型結構。'}, 'short_view': {'structure_type': 'double_bottom', 'structure_range': '2026-01-30 ~ 2026-03-20', 'note': '出現兩個相近低點（約在1/30與2月上旬附近），中間有明確反彈到較高區，第二次回測未明顯跌破前低；其後價格向上推進並回到前次反彈高區附近。'}, 'need_price_data': 'NO', 'need_price_data_reason': ''}, '分析': {'K棒

技術分析工具,解盤內容
K棒,近期K棒以紅黑交錯為主，但在3月中旬出現一段連續紅K推升，伴隨較明顯的長紅棒，顯示追價意願一度升溫。接近3月下旬後，高檔開始出現黑K回吐，且可見上影線，代表上方賣壓開始讓短線攻勢降溫，走勢進入拉抬後的消化階段。
K線圖,價格在先前低位區反覆測試後，右側逐步抬高並向上推進，近期一度接近29元附近後轉為來回震盪。就位置來看，現階段落在前段密集交易區之上緣附近，短線較可能的節奏是先以回檔測試支撐、再決定能否續攻；若回落不破前波抬升的關鍵區，仍有再次上試的空間，反之則容易回到震盪區間內整理。
成交量,近期在上漲段有量能放大，呈現價漲量增的特徵；但在高檔震盪時，量能未能持續擴大，轉為量能起伏、偏向追價力道放緩。若後續價格再上推但量能跟不上，需留意短線出現價量背離的風險；相反地，若回檔時量縮，較有利於支撐守穩。
型態,雙重底（1月30日至3月20日），目前處於型態後的延伸階段，價格已回到前次反彈高點附近，後續重點在於能否站穩並進一步向上突破。
移動平均線,5日線與10日線同步上揚，短線偏多；20日線走平偏上、60日線仍偏下彎，中期偏向盤整。價格目前位於5日線、10日線之上，短線支撐可先看10日線附近約27.8元上下；再往下則看20日線附近約27.3元上下。上方壓力落在近期高點區約29元附近，以及60日線附近約28.8元上下（靠近處容易出現反覆）。
MACD指標,DIF線位於MACD線之上，且兩線走勢向上，顯示動能仍在改善；OSC柱體維持在零軸上方並呈現緩步擴大，代表多方動能仍占優。若後續OSC柱體開始縮小，會是動能降溫的早期訊號。
KD指標,K線與D線位於中高檔區，KD雙線維持向上但斜率趨緩，顯示短線仍偏熱但續強需要新的推力。若K線向下跌破D線形成死亡交叉，較容易引發回檔；反之若回落後快速再度上彎，則代表高檔仍有延續力。
整體評價,整體訊號偏多但帶有分歧：K棒呈現拉抬後高檔出現回吐與上影線，顯示短線攻勢降溫；K線圖位置接近29元附近壓力區，容易先震盪消化；成交量在上漲時有放大但高檔未持續擴量，追價力道需要觀察。另一方面，型態顯示為雙重底後的延伸階段，均線短線偏多且提供27.8元、27.3元附近支撐，MACD動能仍在改善，KD在中高檔但趨緩。 關鍵位階上，短線以29元附近視為主要壓力帶，若能帶量站穩，才有機會打開續攻空間；下方先看27.8元附近（10日線）與27.3元附近（20日線）兩道支撐，跌破其中一層將提高回到原震盪區整理的風險。操作上較適合「守支撐、看突破」：不追高，等待回測支撐的承接力道或突破壓力時的量能表態。
